**Import various packages**

In [ ]:
import time
import torch
import sys
sys.path.append('../input')
sys.path.insert(0, 'Utilities')
import numpy as np
import scipy.io
from scipy.ndimage import zoom
from scipy import linalg
import scipy.sparse as sp
from numpy import inner, conjugate
from numpy.linalg import norm
try:
    from scipy.sparse.linalg.isolve.utils import make_system
except:
    from scipy.sparse.linalg._isolve.utils import make_system
from scipy.sparse.linalg import spsolve
from scipy.sparse.linalg import inv
from scipy.sparse import coo_matrix
import matplotlib.pyplot as plt
from torch.nn import functional as F
import torch.nn as nn
from scipy.sparse.linalg import spsolve
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

**Generation functions of the impedance matrix and source term**

In [ ]:
def matrix_ofd4(nx, nz, h, f, delta, c_vec):
    # formulate the impedance matrix of velocity model by 4-order OFD method
    # 采用4阶OFD法建立速度模型阻抗矩阵
    # 2D rectangle area
    # implement PML condition all around: 4 directions
    # 实现PML条件四周：4个方向
    # last revision: 2021.4.20

    # parameter list
    # omega: angular frequency
    # h: spatial step
    # nx: number of nodes along x direction
    # nz: number of nodes along x direction
    # delta: PML layers

    # imaginary unit
    ii = 1j

    # problem scale
    N = (nx - 2) * (nz - 2)

    # nonzero elements of impedance matrix
    spn = 9 * (nx - 6) * (nz - 6) + 16 * (nx + nz - 12) + 14 * (nx + nz - 12) + 96

    # sparse store: vector space
    ai = np.zeros(spn, dtype=int)
    aj = np.zeros(spn, dtype=int)
    as_ = np.zeros(spn, dtype=complex)
    pa = 0

    # set angular frequency
    omega = 2 * np.pi * f

    # PML parameter: the ratio of reflection
    R = 1e-3

    for k in range(N):
        # grid coordinate conversion: row rule
        # 网格坐标转换：行规则
        j = k // (nx - 2)
        i = k - (nx - 2) * j

        # set velocity: row rule
        c = c_vec[k]

        # set PML attenuation function
        if i < delta - 1:
            dx = (
                    -3 * c / (2 * delta * h) * np.log(R) * ((delta - i - 1) / delta) ** 2
            )
            dxp = -3 * c / (delta * h) ** 2 * np.log(R) * (-(delta - i - 1) / delta)
        elif i > nx - delta - 2:
            dx = (
                    -3 * c / (2 * delta * h) * np.log(R) * ((i - nx + delta + 2) / delta) ** 2
            )
            dxp = -3 * c / (delta * h) ** 2 * np.log(R) * ((i - nx + delta + 2) / delta)
        else:
            dx = 0
            dxp = 0

        if j < delta - 1:
            dz = (
                    -3 * c / (2 * delta * h) * np.log(R) * ((delta - j - 1) / delta) ** 2
            )
            dzp = -3 * c / (delta * h) ** 2 * np.log(R) * (-(delta - j - 1) / delta)
        elif j > nz - delta - 2:
            dz = (
                    -3 * c / (2 * delta * h) * np.log(R) * ((j - nz + delta + 2) / delta) ** 2
            )
            dzp = -3 * c / (delta * h) ** 2 * np.log(R) * ((j - nz + delta + 2) / delta)
        else:
            dz = 0
            dzp = 0

        tx = 1 - ii * dx / omega
        tz = 1 - ii * dz / omega

        # matlab rule: index start from 1
        # matlab规则：指数从1开始
        kt = k + 1

        # 设置关于波场值 u 的行
        # left1
        if i != 0:
            ai[pa] = kt
            aj[pa] = kt - 1
            as_[pa] = 4 / (3 * tx ** 2) - 2 * ii * dxp * h / (3 * omega * tx ** 3)
            pa += 1

        # left2
        if i > 1:
            ai[pa] = kt
            aj[pa] = kt - 2
            as_[pa] = -1 / (12 * tx ** 2) + ii * dxp * h / (12 * omega * tx ** 3)
            pa += 1

        # right1
        if i != nx - 3:
            ai[pa] = kt
            aj[pa] = kt + 1
            as_[pa] = 4 / (3 * tx ** 2) + 2 * ii * dxp * h / (3 * omega * tx ** 3)
            pa += 1

        # right2
        if i < nx - 4:
            ai[pa] = kt
            aj[pa] = kt + 2
            as_[pa] = -1 / (12 * tx ** 2) - ii * dxp * h / (12 * omega * tx ** 3)
            pa += 1

        # up1
        if j != 0:
            ai[pa] = kt
            aj[pa] = kt - nx + 2
            as_[pa] = 4 / (3 * tz ** 2) - 2 * ii * dzp * h / (3 * omega * tz ** 3)
            pa += 1

        # up2
        if j > 1:
            ai[pa] = kt
            aj[pa] = kt - 2 * nx + 4
            as_[pa] = -1 / (12 * tz ** 2) + ii * dzp * h / (12 * omega * tz ** 3)
            pa += 1

        # down1
        if j != nz - 3:
            ai[pa] = kt
            aj[pa] = kt + nx - 2
            as_[pa] = 4 / (3 * tz ** 2) + 2 * ii * dzp * h / (3 * omega * tz ** 3)
            pa += 1

        # down2
        if j < nz - 4:
            ai[pa] = kt
            aj[pa] = kt + 2 * nx - 4
            as_[pa] = -1 / (12 * tz ** 2) - ii * dzp * h / (12 * omega * tz ** 3)
            pa += 1

        # inner
        ai[pa] = kt
        aj[pa] = kt
        as_[pa] = (omega * h / c) ** 2 - 5 / 2 * (1 / tx ** 2 + 1 / tz ** 2)
        pa += 1

    A = coo_matrix((as_, (ai - 1, aj - 1)), shape=(N, N)).tocsr()

    return A

def source_ofd(f, f0, N, h, c_vec, s_loc):
    # generate the right hand side term of source
    # last revision: 2022.7.6

    # initial unit
    ii = 1j

    # source location (count from 0)
    s0 = s_loc

    # source velocity
    sc = c_vec[s0][0]

    # Amplitude and phase
    t0 = 0.12
    Amp = 1e+5

    s = np.sqrt(2) * Amp / (np.pi * f0) * (f / f0) ** 2 * np.exp(-(f / f0) ** 2) * \
        sp.coo_matrix(([-h ** 2 / sc ** 2 * np.exp(-ii * 2 * np.pi * f * t0)], ([s0], [0])), shape=(N, 1))

    return s

**Basic parameter settings for the impendance matrix and source**

In [ ]:
import imageio
import math
from scipy.sparse import diags
from scipy.sparse.linalg import splu,spilu
nx = 66
nz = 66
# spatial step
h = 0.025
# dominant frequency of source
f = 20
f0 = 10
# numbers of PML layers
delta = 10
# c_vec为波速
c_0 = 2 * np.ones(((nz - 2)//2, nx - 2))
c_1 = 2.5 * np.ones(((nz - 2)//2, nx - 2))
cc = np.concatenate((c_0, c_1), axis=0)
v_row_loc = math.ceil((nx - 2) / 2)
v_col_loc = math.ceil((nz - 2) / 3)
v_row_delta = math.ceil((nx - 2) / 10)
v_col_delta = math.ceil((nz - 2) / 3)
cc[v_row_loc:v_row_loc + v_row_delta, v_col_loc:v_col_loc + v_col_delta] = 2
N = (nx-2)*(nz-2)
c_vec = cc.reshape(N, 1)
A = matrix_ofd4(nx, nz, h, f, delta, c_vec)
LU=splu(A)
s_loc_1 = int((nx-2)//2+12*(nz-2))
b = source_ofd(f, f0, N, h, c_vec, s_loc_1).todense()
Gf = np.min(cc)/(2.5*f0*h)

**Set random number seed**

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

**Construction of the training dataset (random version)**

In [ ]:
def generate_random_data(num_samples, matrix, seed=1234):
    nn = (nz-2)//2
    noise = 1e-8
    np.random.seed(seed)
    x_tensor = np.zeros((num_samples, 2, nz-2, nx-2))
    y_tensor = np.zeros((num_samples, 2, nz-2, nx-2))
    for i in range(num_samples):
        x_tensor[i,0,:,:] = 2 * np.random.rand(nz-2, nx-2) - 1
        x_tensor[i,1,:,:] = 2 * np.random.rand(nz-2, nx-2) - 1
        XX = (x_tensor[i,0,:,:] + 1j*x_tensor[i,1,:,:]).reshape(-1, 1)#x为随机得到的
        YY = matrix @ XX#y通过x与A得到的
        y_tensor[i,0,:,:] = YY.real.reshape(x_tensor[i,0,:,:].shape)
        y_tensor[i,1,:,:] = YY.imag.reshape(x_tensor[i,1,:,:].shape)
        power = np.max(np.abs(y_tensor[i]))
        x_tensor[i] = x_tensor[i]/power
        y_tensor[i] = y_tensor[i]/power
    return torch.tensor(x_tensor), torch.tensor(y_tensor)

**Constructing training dataset and validation dataset**

In [ ]:
trainX, trainY = generate_random_data(7000, A)
valX, valY = generate_random_data(700, A)
# Prepare data loaders
train_dataset = TensorDataset(trainY, trainX)
val_dataset = TensorDataset(valY, valX)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=32)

**Network structure of the generator**

In [ ]:
class Conv(nn.Module):
    def __init__(self, C_in, C_out):
        super(Conv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(C_in, C_out, 5, 1, 2),
            nn.Tanh(),
            nn.Conv2d(C_out, C_out, 5, 1, 2),
        )
        self.tanh = nn.Tanh()
        # 添加一个1x1卷积用于匹配维度，仅当输入和输出通道数不相等时使用
        self.match_channels = nn.Conv2d(C_in, C_out, 1, 1, 0) if C_in != C_out else None

    def forward(self, x):
        residual = x
        out = self.conv(x)
        # 如果输入和输出通道数不相等，使用1x1卷积调整残差的通道数
        if self.match_channels is not None:
            residual = self.match_channels(x)
        out += residual  # 添加残差
        return self.tanh(out)  # 应用激活函数

# 下采样模块
class DownSampling(nn.Module):
    def __init__(self, C):
        super(DownSampling, self).__init__()
        self.Down = nn.Sequential(
            # 使用卷积进行2倍的下采样，通道数不变
            nn.Conv2d(C, C, 5, 2, 2),
            nn.Tanh(),
        )
    def forward(self, x):
        return self.Down(x)

# 上采样模块
class UpSampling(nn.Module):
    def __init__(self, C):
        super(UpSampling, self).__init__()
        # 特征图大小扩大2倍，通道数减半
        self.Up = nn.Conv2d(C, C // 2, 1, 1)

    def forward(self, x, r):
        # 使用邻近插值进行下采样
        up = F.interpolate(x, scale_factor=2, mode="nearest-exact")
        x = self.Up(up)
        # 拼接，当前上采样的，和之前下采样过程中的
        return torch.cat((x, r), 1)
    
# 主干网络
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        # 4次下采样
        self.C1 = Conv(2, 64)
        self.D1 = DownSampling(64)
        self.C2 = Conv(64, 128)
        self.D2 = DownSampling(128)
        self.C3 = Conv(128, 256)
        self.D3 = DownSampling(256)
        self.C4 = Conv(256, 512)
        self.D4 = DownSampling(512)
        self.C5 = Conv(512, 1024)
        # 3次上采样
        self.U1 = UpSampling(1024)
        self.C6 = Conv(1024, 512)
        self.U2 = UpSampling(512)
        self.C7 = Conv(512, 256)
        self.U3 = UpSampling(256)
        self.C8 = Conv(256, 128)
        self.U4 = UpSampling(128)
        self.C9 = Conv(128, 64)
        self.Th = nn.Tanh()
        self.pred = torch.nn.Conv2d(64, 2, 5, 1, 2)

    def forward(self, x):
        # 下采样部分
        R1 = self.C1(x)
        R2 = self.C2(self.D1(R1))
        R3 = self.C3(self.D2(R2))
        R4 = self.C4(self.D3(R3))
        Y1 = self.C5(self.D4(R4))
        # 上采样部分
        O1 = self.C6(self.U1(Y1, R4))
        O2 = self.C7(self.U2(O1, R3))
        O3 = self.C8(self.U3(O2, R2))
        O4 = self.C9(self.U4(O3, R1))
        return self.Th(self.pred(O4))

Network structure of the discriminator

In [ ]:
class PatchGAN(nn.Module):
    def __init__(self):
        super(PatchGAN, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(4, 64, 4, stride=2, padding=1),
            nn.Tanh(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.Tanh(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.Tanh(),
            nn.Conv2d(256, 512, 4, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.Tanh(),
            nn.Conv2d(512, 1, kernel_size=4),
            #nn.AdaptiveAvgPool2d(1),  # 添加全局平均池化层
            nn.Sigmoid()
        )

        
    def forward(self, x, y):
        x = torch.cat([x, y], axis=1)
        x = self.model(x)
        return x.view(x.size(0), -1)  # 调整输出形状以匹配目标标签的尺寸

**Network training**

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Create the model
netD = PatchGAN().to(device)
netG = UNet().to(device)
optimizerD = optim.Adam(netD.parameters(), lr=1e-4,betas=(0.5,0.999))
optimizerG = optim.AdamW(netG.parameters(), lr=1e-5)
scheduler = StepLR(optimizerG, step_size=50, gamma=0.5)

BCE_Loss = nn.BCEWithLogitsLoss()
L1_Loss = nn.L1Loss()
L2_Loss = nn.MSELoss()

# 训练GAN
num_epochs = 150

# Initialize lists to monitor loss and accuracy
d_losses = []
g_losses = []
L1es=[]
val_d_losses1 = []
val_d_losses2 = []
val_g_losses1 = []
val_g_losses2 = []
for epoch in range(num_epochs):
    for i, (yi, xi) in enumerate(train_loader):
        xi = xi.float().to(device)
        yi = yi.float().to(device)
        #for ii in range(1):
            # Train Discriminator
        optimizerD.zero_grad()
        D_real = netD(xi, yi)
        D_real_label = torch.ones_like(D_real)
        D_real_loss = BCE_Loss(D_real, D_real_label)
        x_fake = netG(yi)
        D_fake = netD(x_fake, yi).detach()
        D_fake_loss = BCE_Loss(D_fake, torch.zeros_like(D_fake))
        D_loss = (D_real_loss + D_fake_loss) / 2
        D_loss.backward()
        optimizerD.step()

        #for ii in range(1):
            # Train Generator
        optimizerG.zero_grad()
        D_fake = netD(x_fake, yi)
        G_fake_loss = BCE_Loss(D_fake, torch.ones_like(D_fake))
        L1 = L1_Loss(xi, x_fake)
        G_loss = G_fake_loss*1 + L1*100
        G_loss.backward()
        optimizerG.step()
    scheduler.step()

     # 验证
    val_d_loss1 = 0.0
    val_d_loss2 = 0.0
    val_g_loss1 = 0.0
    val_g_loss2 = 0.0
    with torch.no_grad():
        for val_yi, val_xi in val_loader:
            val_xi = val_xi.float().to(device)
            val_yi = val_yi.float().to(device)
            val_x_fake = netG(val_yi)
            val_d_loss1 += BCE_Loss(netD(val_xi, val_yi), torch.ones_like(netD(val_xi, val_yi))).item()
            val_d_loss2 += BCE_Loss(netD(val_x_fake, val_yi), torch.zeros_like(netD(val_xi, val_yi))).item()
            val_g_loss1 += BCE_Loss(netD(val_x_fake, val_yi), torch.ones_like(netD(val_xi, val_yi))).item()
            val_g_loss2 += L1_Loss(val_xi, val_x_fake).item()
        val_d_loss1 /= len(val_loader)
        val_d_loss2 /= len(val_loader)
        val_g_loss1 /= len(val_loader)
        val_g_loss2 /= len(val_loader)
        val_d_losses1.append(val_d_loss1)
        val_d_losses2.append(val_d_loss2)
        val_g_losses1.append(val_g_loss1)
        val_g_losses2.append(val_g_loss2)

    d_losses.append(D_loss.item())
    g_losses.append(G_loss.item())
    L1es.append(L1.item())
torch.save(netG.state_dict(), '/kaggle/working/netG_weights.pth')  # 保存生成器

**Save the trained network parameters**

In [ ]:
torch.save(model.state_dict(),'Sunken_netG_weights.pth')